In [1]:
#Multiprocessing for Parallelism
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 

import holidays
import time
from dateutil.tz import *

from sklearn.neural_network import  MLPRegressor
from sklearn.linear_model import SGDRegressor
from sklearn.inspection import permutation_importance

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

import warnings
warnings.filterwarnings('ignore')

In [2]:
start_time = time.monotonic()

In [3]:
import random

random.seed(42)
np.random.seed(42)

In [4]:
import calendar

def findDay(date):
	year, month, day = (int(i) for i in date.split('-')) 
	return calendar.weekday(year, month, day)

In [11]:
def split_and_preprocess(df):
        clients_data = []
        clients_test = []
        clients_scalers = []
        clients_imputers = []  # optional, for reproducibility

        clients_test_original = []  # (X_test_original_i, Y_test_original_i)
        clients_val_original = []  # (X_val_original_i, Y_val_original_i)

        #for sub_df in df:
        X, Y = make_XY(df)

        X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, shuffle=False)
        X_train2, X_val, Y_train2, Y_val = train_test_split(X_train, Y_train, test_size=0.2, shuffle=False)

        # 1) Impute (fit only on training)
        imputer_X = SimpleImputer(strategy="median")
        X_train_imp = imputer_X.fit_transform(X_train)
        X_val_imp   = imputer_X.transform(X_val)
        X_test_imp  = imputer_X.transform(X_test)

        # impute first if needed, then scale (as you already do)
        scaler_X = StandardScaler()
        X_train_scaled = scaler_X.fit_transform(X_train_imp)
        X_val_scaled   = scaler_X.transform(X_val_imp)
        X_test_scaled  = scaler_X.transform(X_test_imp)

        scaler_Y = StandardScaler()
        Y_train_scaled = scaler_Y.fit_transform(Y_train.reshape(-1,1)).ravel()
        Y_val_scaled   = scaler_Y.transform(Y_val.reshape(-1,1)).ravel()
        Y_test_scaled  = scaler_Y.transform(Y_test.reshape(-1,1)).ravel()

        # save scaled for training/testing
        clients_data.append((X_train_scaled, Y_train_scaled))
        clients_test.append((X_test_scaled, Y_test_scaled))
        clients_scalers.append((scaler_X, scaler_Y))
        clients_imputers.append(imputer_X)

        # save ORIGINAL (per client)
        X_test_original_i = scaler_X.inverse_transform(X_test_scaled)
        Y_test_original_i = scaler_Y.inverse_transform(Y_test_scaled.reshape(-1,1)).ravel()
        clients_test_original.append((X_test_original_i, Y_test_original_i))

        X_val_original_i = scaler_X.inverse_transform(X_val_scaled)
        Y_val_original_i = scaler_Y.inverse_transform(Y_val_scaled.reshape(-1,1)).ravel()
        clients_val_original.append((X_val_original_i, Y_val_original_i))
        return (clients_data, clients_test, clients_scalers, clients_test_original, clients_val_original)

In [ ]:
def train_SGD_client_proc(client_data_tuple): 
    model = SGDRegressor(penalty="l2", #  loss='huber'
                        alpha=1e-4,
                        learning_rate="adaptive",
                        eta0=1e-3,
                        max_iter=2000,
                        tol=1e-4,
                        early_stopping=True,
                        validation_fraction=0.1,
                        n_iter_no_change=20,
                        random_state=42
                    ).fit(client_data_tuple[0],client_data_tuple[1])
    return model

In [ ]:
def train_SGD_model(clients_data):
    num_of_clients = len(clients_data)
    print(f"Training {num_of_clients} shards in parallel...")
    max_workers = num_of_clients

    # Each client trains its own SGD model
    client_models = []
    # Train all clients in parallel
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = [
                executor.submit(train_SGD_client_proc, clients_data[j])
                for j in range(num_of_clients)
            ]
            for f in as_completed(futures):
                client_models.append(f.result())
    return client_models

In [ ]:
# Extract weights from a trained client model
# SGDRegressor stores learned layer weights in coefs_ and biases in intercepts_[0].
def get_SGD_model_weights(model):
    return {
        "coef": model.coef_.copy(),
        "intercept": np.array([model.intercept_[0]])
    }

In [ ]:
# Put averaged weights into a server/global model
def average_SGD_model_weights(client_weights):
    avg_coef = np.mean([w["coef"] for w in client_weights], axis=0)
    avg_intercept = np.mean([w["intercept"] for w in client_weights], axis=0)
    return {"coef": avg_coef, "intercept": avg_intercept}

In [ ]:
# Put averaged weights into a server/global model
def build_global_SGD_model_like(client_models, X_sample, y_sample):
    """
    Create a model with same architecture as client_models[0]
    and initialize internal structures via a dummy fit.
    """
    base = client_models[0]

    global_model = SGDRegressor(
        random_state=base.random_state
    )

    # fit to initialize shapes
    global_model.fit(X_sample[:2], y_sample[:2])

    return global_model

In [ ]:
# Assign averaged weights to the global model
def set_SGD_model_weights(model, weights):
    model.coef_ = weights["coef"].copy()
    model.intercept_ = weights["intercept"].copy()
    return model

In [ ]:
# Assign averaged weights to the global model
def fedavg_sgd(client_models, X_sample, y_sample):
    # extract
    #print("Extracting client model weights...")
    client_weight_list = [get_SGD_model_weights(m) for m in client_models]

    # average
    avg_weights = average_SGD_model_weights(client_weight_list)

    # build server/global model
    global_model = build_global_SGD_model_like(client_models, X_sample, y_sample)

    # load averaged weights
    global_model = set_SGD_model_weights(global_model, avg_weights)

    return global_model

In [ ]:
def infer_with_global_SGD_model(global_model, clients_test, clients_scalers):
    prediction_per_shard = []
    prediction_time_per_shard = []
    ytest_per_shard = []

    for i, (X_test_i, Y_test_i_scaled) in enumerate(clients_test):
        start = time.perf_counter()
        _, scaler_Y = clients_scalers[i]

        pred_i_scaled = global_model.predict(X_test_i).reshape(-1, 1)
        #loss_curve_i = global_model.loss_curve_ 
        pred_i = scaler_Y.inverse_transform(pred_i_scaled).reshape(-1)
        pred_i = np.clip(pred_i, 0.0, None)

        # also reconstruct Y_test in original units if you want plots/metrics
        y_i = scaler_Y.inverse_transform(Y_test_i_scaled.reshape(-1, 1)).reshape(-1)

        prediction_per_shard.append(pred_i)
        ytest_per_shard.append(y_i)
        pred_time_s = time.perf_counter() - start
        prediction_time_per_shard.append(pred_time_s)

    return prediction_per_shard, prediction_time_per_shard 

In [ ]:
def locally_infer_from_SGD_model(client_models, clients_test, clients_scalers):
    prediction_per_shard = []
    prediction_time_per_shard = []
    loss_curves = []
    ytest_per_shard = []

    for i, (X_test_i, Y_test_i_scaled) in enumerate(clients_test):
        start = time.perf_counter()
        _, scaler_Y = clients_scalers[i]

        pred_i_scaled = client_models[i].predict(X_test_i).reshape(-1, 1)
        pred_i = scaler_Y.inverse_transform(pred_i_scaled).reshape(-1)
        pred_i = np.clip(pred_i, 0.0, None)

        # also reconstruct Y_test in original units if you want plots/metrics
        y_i = scaler_Y.inverse_transform(Y_test_i_scaled.reshape(-1, 1)).reshape(-1)

        prediction_per_shard.append(pred_i)
        ytest_per_shard.append(y_i)
        pred_time_s = time.perf_counter() - start
        prediction_time_per_shard.append(pred_time_s)

    return prediction_per_shard, prediction_time_per_shard 

In [21]:
def compute_error(Y_test_original_all, pred):
    mae = np.round(mean_absolute_error(Y_test_original_all, pred),4)
    mse = mean_squared_error(Y_test_original_all, pred)
    rmse = np.round(mse ** 0.5, 4)
    r2 = np.round(r2_score(Y_test_original_all, pred),4)
    return mae, mse, rmse, r2

In [ ]:
# -----------------------------
# Helper: build shards
# -----------------------------

def _cluster_shards(num_shards):
    
    return num_shards

# -----------------------------
# Helper: build shards
# -----------------------------
def _random_shards(hh_ids, num_shards, rng):
    """
    Randomly permute households and split into `num_shards` groups
    as evenly as possible (for your cases, sizes are exact).
    Returns: List[List[hh_id]]
    """
    print(f"Creating {num_shards} random shards from {len(hh_ids)} households...")
    hh_ids = list(hh_ids)
    perm = rng.permutation(hh_ids).tolist()
    return [perm[i::num_shards] for i in range(num_shards)]


# -----------------------------
# Helper: aggregate data per shard
# -----------------------------
def _aggregate_households(datasets_by_hh, shard_hh_ids):
    """
    Given mapping {hh_id: df_or_data}, aggregate the households listed in shard_hh_ids
    into one shard dataset.
    You will implement actual aggregation (concat, sort by time, feature engineering, etc.).
    """
    shard_frames = []

    for hid in shard_hh_ids:
        df = datasets_by_hh[hid].copy()
        df["hh_id"] = hid
        shard_frames.append(df)

    shard_df = pd.concat(shard_frames, axis=0, ignore_index=True)

    # optional: keep temporal order if needed
    # shard_df = shard_df.sort_values(["hh_id", "Year", "Month", "Weekday", "Hour", "Minute"])

    return shard_df


# -----------------------------
# Helper: SGD train + aggregate + infer + error
# -----------------------------
def _SGD_train_aggregate_infer_error(shard_datasets, shard_hh_ids, type, features_cols):

    all_clients_data = []
    all_clients_test = []
    all_clients_scalers = []
    all_clients_test_original = []
    all_clients_val_original = []

    # process each shard separately
    for shard_dataset in shard_datasets:

        clients_data, clients_test, clients_scalers, clients_test_original, clients_val_original = split_and_preprocess(shard_dataset)

        all_clients_data.extend(clients_data)
        all_clients_test.extend(clients_test)
        all_clients_scalers.extend(clients_scalers)
        all_clients_test_original.extend(clients_test_original)
        all_clients_val_original.extend(clients_val_original)

    if type == "Federated":
        # train models
        client_models = train_SGD_model(all_clients_data)
        
        # aggregate models
        global_model = fedavg_sgd(client_models, all_clients_data[0][0], all_clients_data[0][1])

        # predict with global model
        start = time.perf_counter()
        prediction_per_shard, prediction_time_per_shard = infer_with_global_SGD_model(
            global_model , all_clients_test, all_clients_scalers
        )
        prediction_time_total = time.perf_counter() - start
        r2_scores = []
        mse_num_shards = []
        for j, model in enumerate(client_models):
            X_val, Y_val = all_clients_val_original[j]
            Y_pred = model.predict(X_val)
            mse_num_shards.append(mean_squared_error(Y_val, Y_pred))
            r2_scores.append(r2_score(Y_val, Y_pred))
        print("Federated - MSE: ", len(mse_num_shards), " ", (mse_num_shards))
        print("Federated - R2: ", len(r2_scores), " ", (r2_scores))
        
    elif type == "Local":
        # train models
        client_models = train_SGD_model(all_clients_data)
        
        # predict with global model
        start = time.perf_counter()
        prediction_per_shard, prediction_time_per_shard = locally_infer_from_SGD_model(
            client_models, all_clients_test, all_clients_scalers
        )
        prediction_time_total = time.perf_counter() - start
        r2_scores = []
        mse_num_shards = []
        for j, model in enumerate(client_models):
            X_val, Y_val = all_clients_val_original[j]
            Y_pred = model.predict(X_val)
            mse_num_shards.append(mean_squared_error(Y_val, Y_pred))
            r2_scores.append(r2_score(Y_val, Y_pred))
        print("Local - MSE: ", len(mse_num_shards), " ", (mse_num_shards))
        print("Local - R2: ", len(r2_scores), " ", (r2_scores))
    elif type == "Centralized":
        # train model on all data
        X_all = np.concatenate([cd[0] for cd in all_clients_data])
        Y_all = np.concatenate([cd[1] for cd in all_clients_data])
        global_model = SGDRegressor(random_state=42).fit(X_all, Y_all)
        print("Trained centralized SGD model.")

        # predict with global model
        start = time.perf_counter()
        prediction_per_shard, prediction_time_per_shard = infer_with_global_SGD_model(
            global_model , all_clients_test, all_clients_scalers
        )
        prediction_time_total = time.perf_counter() - start
        r2_scores = []
        mse_num_shards = [] 
        for client_val_data in all_clients_val_original:
            X_val, Y_val = client_val_data
            Y_pred = global_model.predict(X_val)
            mse_num_shards.append(mean_squared_error(Y_val, Y_pred))
            r2_scores.append(r2_score(Y_val, Y_pred))
        print("Centralized - MSE: ", len(mse_num_shards), " ", (mse_num_shards))
        print("Centralized - R2: ", len(r2_scores), " ", (r2_scores))


    mae, mse, rmse, r2 = [], [], [], []
    for i, (Xo, Yo) in enumerate(all_clients_test_original):
        mae_val, mse_val, rmse_val, r2_val = compute_error(
                                                            Yo, 
                                                            prediction_per_shard[i]
                                                            )
        mae.append(mae_val)
        mse.append(mse_val)
        rmse.append(rmse_val)
        r2.append(r2_val)
    
    return {
        "shard_hh_ids": shard_hh_ids,
        "mae": mae,
        "mse": mse,
        "rmse": rmse,
        "r2": r2,
        "prediction_per_shard": ((prediction_per_shard)),
        "prediction_time_per_shard": np.round(np.array(prediction_time_per_shard), 4),
        "prediction_time_total": np.round(float(prediction_time_total), 4),
        "clients_test_original": all_clients_test_original
    }

# -----------------------------
# Scenario runners
# -----------------------------
def Federated_shards(datasets_by_hh, model, run_id, rng, num_shards, keep_cols=None):
    """
    num_shards -> num_shards HHs isolated (12/num_shards HH per shard)
    """
    start = time.perf_counter()
    hh_ids = list(datasets_by_hh.keys())
    if rng is not None:
        shards = _random_shards(hh_ids, num_shards=num_shards, rng=rng)  # will produce 12 groups of size 1
    elif rng is None:
        shards = _cluster_shards(num_shards=num_shards)  # will produce 12 groups of size 1
    print(f"Shards (HH IDs): {shards}")
    shard_datasets = []
    shard_hh_ids_list = []
    errors = []
    rows = []  # list of DataFrames, concat once at the end
    prediction_time_total = []
    prediction_time_per_shard = []
    for shard_hh_ids in shards:
        shard_dataset = _aggregate_households(datasets_by_hh, shard_hh_ids)
        shard_hh_ids_list.append(shard_hh_ids)
        shard_datasets.append(shard_dataset)

    if keep_cols is None:
        features_cols = ["air1_lag_1h",  "furnace1_lag_1h", "usage_lag_1h", "Month","Weekday", "Type_of_day","Hour"] #,"Minute","furnace1_lag_48h","usage_lag_48h",

    else:
        features_cols = [f"x{c}" for c in keep_cols]

    if  model == "SGD":
        result = _SGD_train_aggregate_infer_error(shard_datasets, shard_hh_ids_list, type="Federated", features_cols=features_cols)
    prediction_time_per_shard.append(result["prediction_time_per_shard"])
    prediction_time_total.append(result["prediction_time_total"])
    errors.append({
            "mae": result["mae"],
            "mse": result["mse"],
            "rmse": result["rmse"],
            "r2": result["r2"]
        })

    for i, shard_hh_ids in enumerate(shard_hh_ids_list):
        Xo, Yo = result["clients_test_original"][i]
        prediction_per_shard = result["prediction_per_shard"][i]

        Xo = np.asarray(Xo)
        Yo = np.asarray(Yo).reshape(-1)
        prediction_per_shard = np.asarray(prediction_per_shard).reshape(-1)

        if keep_cols is None:
            cols =  ["air1_lag_1h",  "furnace1_lag_1h", "usage_lag_1h", "Month","Weekday", "Type_of_day","Hour"]
            df_tmp = pd.DataFrame(Xo, columns=cols)
        else:
            df_tmp = pd.DataFrame(Xo[:, keep_cols], columns=[f"x{c}" for c in keep_cols])

        df_tmp["hh_id"] = str(shard_hh_ids)  # good for multi-HH shards
        df_tmp["Actual"] = Yo
        df_tmp["Predicted"] = prediction_per_shard

        rows.append(df_tmp)

    pred_test_df = pd.concat(rows, ignore_index=True)
    completion_time_s = time.perf_counter() - start
    return errors, pred_test_df, prediction_time_per_shard, prediction_time_total, np.round(float(completion_time_s), 4) #, loss_curves_shards 

def Centralized(datasets_by_hh, model, run_id, rng=None, keep_cols=None):
    """
    Centralized -> all 12 HHs aggregated into one dataset, one model.
    rng not used (kept for interface symmetry).
    """
    start = time.perf_counter()
    hh_ids = list(datasets_by_hh.keys())
    shard_dataset = _aggregate_households(datasets_by_hh, hh_ids)
    
    errors = []
    rows = []  # list of DataFrames, concat once at the end
    prediction_time_per_shard = []
    prediction_time_total = []

    if keep_cols is None:
        features_cols = ["air1_lag_1h",  "furnace1_lag_1h", "usage_lag_1h", "Month","Weekday", "Type_of_day","Hour"]
    else:
        features_cols = [f"x{c}" for c in keep_cols]

    if model == "SGD":
        result = _SGD_train_aggregate_infer_error([shard_dataset], [hh_ids], type="Centralized", features_cols=features_cols)

    prediction_time_per_shard.append(result["prediction_time_per_shard"])
    prediction_time_total.append(result["prediction_time_total"])
    errors.append({
            "mae": result["mae"],
            "mse": result["mse"],
            "rmse": result["rmse"],
            "r2": result["r2"]
        })

    Xo, Yo = result["clients_test_original"][0]
    prediction_per_shard = result["prediction_per_shard"][0]

    Xo = np.asarray(Xo)
    Yo = np.asarray(Yo).reshape(-1)
    prediction_per_shard = np.asarray(prediction_per_shard).reshape(-1)

    if keep_cols is None:
        cols = ["air1_lag_1h",  "furnace1_lag_1h", "usage_lag_1h", "Month","Weekday", "Type_of_day","Hour"]
        df_tmp = pd.DataFrame(Xo, columns=cols)
    else:
        df_tmp = pd.DataFrame(Xo[:, keep_cols], columns=[f"x{c}" for c in keep_cols])

    df_tmp["hh_id"] = "Centralized"
    df_tmp["Actual"] = Yo
    df_tmp["Predicted"] = prediction_per_shard

    rows.append(df_tmp)
    pred_test_df = pd.concat(rows, ignore_index=True)
    
    completion_time_s = time.perf_counter() - start
    print(f"Completed in {completion_time_s:.4f} seconds")
    return errors, pred_test_df, prediction_time_per_shard, prediction_time_total, np.round(float(completion_time_s), 4) #, result["loss_curves"]
    
def Localized(datasets_by_hh, model, run_id, rng=None, keep_cols=None):
    """
    Localized -> each HH is its own shard, 12 models, no aggregation.
    rng not used (kept for interface symmetry).
    """
    start = time.perf_counter()
    hh_ids = list(datasets_by_hh.keys())
    
    errors = []
    rows = []  # list of DataFrames, concat once at the end
    prediction_time_per_shard = []
    prediction_time_total = []

    if keep_cols is None:
        features_cols =  ["air1_lag_1h",  "furnace1_lag_1h", "usage_lag_1h", "Month","Weekday", "Type_of_day","Hour"]
    else:
        features_cols = [f"x{c}" for c in keep_cols]

    for hh_id in hh_ids:
        shard_dataset = _aggregate_households(datasets_by_hh, [hh_id])
        if model == "SGD":
            result = _SGD_train_aggregate_infer_error([shard_dataset], [[hh_id]], type="Local", features_cols=features_cols)

        prediction_time_per_shard.append(result["prediction_time_per_shard"])
        prediction_time_total.append(result["prediction_time_total"])
        errors.append({
                "mae": result["mae"],
                "mse": result["mse"],
                "rmse": result["rmse"],
                "r2": result["r2"]
            })

        Xo, Yo = result["clients_test_original"][0]
        prediction_per_shard = result["prediction_per_shard"][0]

        Xo = np.asarray(Xo)
        Yo = np.asarray(Yo).reshape(-1)
        prediction_per_shard = np.asarray(prediction_per_shard).reshape(-1)

        if keep_cols is None:
            cols = ["air1_lag_1h",  "furnace1_lag_1h", "usage_lag_1h", "Month","Weekday", "Type_of_day","Hour"] #,"Minute","furnace1_lag_48h","usage_lag_48h",
            df_tmp = pd.DataFrame(Xo, columns=cols)
        else:
            df_tmp = pd.DataFrame(Xo[:, keep_cols], columns=[f"x{c}" for c in keep_cols])

        df_tmp["hh_id"] = str(hh_id)
        df_tmp["Actual"] = Yo
        df_tmp["Predicted"] = prediction_per_shard

        rows.append(df_tmp)

    pred_test_df = pd.concat(rows, ignore_index=True)
    
    completion_time_s = time.perf_counter() - start
    print(f"Completed in {completion_time_s:.4f} seconds")
    return errors, pred_test_df, prediction_time_per_shard, prediction_time_total, np.round(float(completion_time_s), 4) #, result["loss_curves"]

# -----------------------------
# Experiment: repeat 100 times
# -----------------------------
def run_experiments(datasets_by_hh, model, seed, repeats):
    """
    Repeats each scenario 100 times:
      - random HH permutation
      - shard assignment
      - aggregation
      - train
      - aggregate (FedAvg)
      - infer
      - error

    Returns a dict of results you can post-process (mean/std, boxplots, etc.).
    """

    if seed is not None:
        rng = np.random.default_rng(seed)
    elif seed is None:
        rng = None

    results = {
        "Cent.": {},
        "Local.": {},
        "Fed_2shards": {},
        "Fed_3shards": {},
        "Fed_4shards": {},
        "Fed_6shards": {},
        "Fed_12shards": {}
    }


    for run_id in range(repeats):
        results[f"Cent."][run_id] = Centralized(
                datasets_by_hh, model, run_id, rng=None, keep_cols=None
            )
        results[f"Local."][run_id] = Localized(
                datasets_by_hh, model, run_id, rng=None, keep_cols=None
            )
        for i in [2,3,4,6,12]:
            results[f"Fed_{i}shards"][run_id] = Federated_shards(
                datasets_by_hh, model, run_id, rng, i, keep_cols=None
            )
    print(f"All experiments of {model} with seed {seed} is completed.")
    return results

In [ ]:
datasets_by_hh = {
    }

In [ ]:
results_rand_SGD = run_experiments(datasets_by_hh, "SGD", seed=0, repeats=3)

Trained centralized SGD model.
Centralized - MSE:  1   [1.4312190265297153]
Centralized - R2:  1   [0.3098367839200453]
Completed in 40.3312 seconds
Training 1 shards in parallel...
Local - MSE:  1   [1.872230909276377]
Local - R2:  1   [-36.68825319684368]
Training 1 shards in parallel...
Local - MSE:  1   [1.1949585871946102]
Local - R2:  1   [-3.518288550012267]
Training 1 shards in parallel...
Local - MSE:  1   [0.12806915999725163]
Local - R2:  1   [-0.07035443267495878]
Training 1 shards in parallel...
Local - MSE:  1   [0.7445637585221655]
Local - R2:  1   [-11.64969980024109]
Training 1 shards in parallel...
Local - MSE:  1   [0.13102757242775787]
Local - R2:  1   [-11.960743739621504]
Training 1 shards in parallel...
Local - MSE:  1   [0.7657980949461516]
Local - R2:  1   [-2.363770919335102]
Training 1 shards in parallel...
Local - MSE:  1   [0.6587472996444121]
Local - R2:  1   [-0.9369124335127039]
Training 1 shards in parallel...
Local - MSE:  1   [0.9463920589963445]
Loca

In [ ]:
results_sim_SGD = run_experiments(datasets_by_hh, "SGD", seed=None, repeats=3)

Trained centralized SGD model.
Centralized - MSE:  1   [1.4312190265297153]
Centralized - R2:  1   [0.3098367839200453]
Completed in 27.3264 seconds
Training 1 shards in parallel...
Local - MSE:  1   [1.872230909276377]
Local - R2:  1   [-36.68825319684368]
Training 1 shards in parallel...
Local - MSE:  1   [1.1949585871946102]
Local - R2:  1   [-3.518288550012267]
Training 1 shards in parallel...
Local - MSE:  1   [0.12806915999725163]
Local - R2:  1   [-0.07035443267495878]
Training 1 shards in parallel...
Local - MSE:  1   [0.7445637585221655]
Local - R2:  1   [-11.64969980024109]
Training 1 shards in parallel...
Local - MSE:  1   [0.13102757242775787]
Local - R2:  1   [-11.960743739621504]
Training 1 shards in parallel...
Local - MSE:  1   [0.7657980949461516]
Local - R2:  1   [-2.363770919335102]
Training 1 shards in parallel...
Local - MSE:  1   [0.6587472996444121]
Local - R2:  1   [-0.9369124335127039]
Training 1 shards in parallel...
Local - MSE:  1   [0.9463920589963445]
Loca